# Train HeatGeo from GitHub (Colab)

Notebook này luôn đồng bộ code mới nhất từ branch `nqd_mass_geom_loss` trước khi cài dependencies và train. Chọn một teacher–student pair rồi dùng **Runtime → Run all**.

## `L_row` sau khi rút gọn

Mỗi pool column được teacher chọn (diffusion support — không tính hard/uniform negative) được promote thành một auxiliary row, và khớp toàn bộ transition row khả dụng của nó với **trọng số đều**. Batch anchor bị loại vì `L_rel` đã khớp transition row của chúng ở scale r=1. Row set là hàm **tất định** của candidate pool, nên `L_row` không còn hyperparameter chọn row nào — chỉ còn `ROW_WEIGHT`.

## Bốn biến thể đã đo và đã xóa

Tất cả ở `qwen3_0_6b_to_minilmv2_h384`, seed 42, 5 epochs, `ROW_WEIGHT=1.0`. Code nằm trong git history tại `b1f683b` trở về trước.

| Biến thể | Avg | Vì sao bỏ |
|---|---:|---|
| **uniform (giữ lại)** | **74.86** / 74.82 | start epoch 2 / 1 |
| non-backtracking walk | 74.88 | ngang uniform nhưng tốn `num_walks`+`walk_length`; sampler phải chèn node vào draw |
| trọng số theo exposed mass | 74.76 | bình phương một selection bias vốn đã tỷ lệ mass |
| trọng số 1/c_B(j) | trơ | c_B(j)=1 cho ~99% row ở corpus/batch này |
| ambient r=0 cho mỗi row | 74.62 | −0.30 out-of-domain, đúng benchmark nó nhắm tới |

Dòng cuối cũng đóng luôn một caveat: `row_exposed_mass = 0.44` (restricted target chỉ chuẩn hóa trên 44% mass thật) **không** phải điểm yếu thực tế — bù calibration cho nó làm kết quả tệ đi.

## Còn lại để thử (chưa chạy)

| Run | Thay đổi | Giả thuyết |
|---|---|---|
| A1 | `ROW_WEIGHT = 1.5` | vùng α>0.5 ở dạng tổng-weight-1 chưa thăm dò; bảng monotone tới đúng mép |
| A2 | `LEARNING_RATE = 3e-5` | underfitting trong cap 5 epoch: mọi KL còn giảm ở epoch 5, `student_top1` 0.21 vs teacher 0.33 |

Chạy **tách riêng từng arm** — run stack vừa rồi cho thấy nếu không có metric tách bạch thì không quy được nguyên nhân. `RUN_TAG` mang `ROW_WEIGHT`, `ROW_START_EPOCH`, `LEARNING_RATE` và `SEED` nên các run không đè lên nhau. Số cuối cho paper phải là mean 3 seed.

> Lưu ý: cell đồng bộ code dùng `git reset --hard` bên trong `/content/embedding-kd`, nên các chỉnh sửa tracked cục bộ trong bản clone Colab sẽ bị bỏ. Cache và output untracked không bị xóa. Pair Qwen3-4B → BERT-base cần GPU có VRAM lớn.


In [29]:
#@title 1. Cấu hình experiment
REPO_URL = "https://github.com/Savoxism/embedding-kd.git"
BRANCH = "nqd_mass_geom_loss"

PAIR_KEY = "qwen3_0_6b_to_minilmv2_h384" #@param
# ["qwen3_0_6b_to_minilmv2_h384", "bge_m3_to_minilmv2_h768", "qwen3_4b_to_bert_base"]

# L_row chỉ còn một knob. Row selection là hàm tất định của candidate pool:
# mọi pool column được teacher chọn đều thành auxiliary row, trọng số đều.
ROW_WEIGHT = 1.0 #@param {type:"number"}
ROW_START_EPOCH = 1 #@param {type:"integer"}

RUN_NAME = "row_loss_run" #@param {type:"string"}

BATCH_SIZE = 64 #@param {type:"integer"}
EPOCHS = 5 #@param {type:"integer"}
LEARNING_RATE = 2e-5 #@param {type:"number"}
MAX_LENGTH = 256 #@param {type:"integer"}
SEED = 42 #@param {type:"integer"}
NUM_WORKERS = 4 #@param {type:"integer"}
TRAIN_DATA = "data/train_set/merged_3_data_5k_each.csv" #@param {type:"string"}

USE_WANDB = False #@param {type:"boolean"}
WANDB_PROJECT = "iclr-mdd-heatgeo" #@param {type:"string"}
WANDB_MODE = "offline" #@param ["online", "offline", "disabled"]
FINAL_WEIGHTS_ONLY = True #@param {type:"boolean"}
REQUIRE_GPU = True #@param {type:"boolean"}

USE_GOOGLE_DRIVE = False #@param {type:"boolean"}
DRIVE_ROOT = "/content/drive/MyDrive/embedding-kd-runs" #@param {type:"string"}

# Thêm CLI flags nếu cần, ví dụ: --graph_k 100 --diffusion_quota 20
EXTRA_ARGS = "" #@param {type:"string"}

assert ROW_WEIGHT >= 0, "ROW_WEIGHT phải không âm"
assert ROW_START_EPOCH >= 1, "ROW_START_EPOCH phải bắt đầu từ 1"
assert BATCH_SIZE > 0 and EPOCHS > 0 and MAX_LENGTH > 0

# Mỗi arm ghi vào thư mục riêng để các run không đè lên nhau. Phải mang đủ mọi
# knob đang quét, kể cả learning rate.
RUN_TAG = f"{RUN_NAME}_w{ROW_WEIGHT:g}_e{ROW_START_EPOCH}_lr{LEARNING_RATE:g}_seed{SEED}"


In [30]:
# 2. Clone lần đầu; các lần Run all sau luôn fetch/reset/pull branch mới nhất
from pathlib import Path
import subprocess

REPO_DIR = Path("/content/embedding-kd")

def git(*args):
    command = ["git", "-C", str(REPO_DIR), *args]
    print("+", " ".join(command))
    subprocess.run(command, check=True)

if (REPO_DIR / ".git").is_dir():
    git("remote", "set-url", "origin", REPO_URL)
    # Bỏ thay đổi tracked trong clone Colab để luôn checkout được remote head.
    git("reset", "--hard")
    git("fetch", "--prune", "origin")
    git("checkout", "-B", BRANCH, f"origin/{BRANCH}")
    git("reset", "--hard", f"origin/{BRANCH}")
    git("pull", "--ff-only", "origin", BRANCH)
elif REPO_DIR.exists() and any(REPO_DIR.iterdir()):
    raise RuntimeError(f"{REPO_DIR} tồn tại nhưng không phải Git repository")
else:
    subprocess.run(
        ["git", "clone", "--branch", BRANCH, "--single-branch", REPO_URL, str(REPO_DIR)],
        check=True,
    )

commit = subprocess.check_output(
    ["git", "-C", str(REPO_DIR), "rev-parse", "HEAD"], text=True
).strip()
print(f"Ready: {BRANCH}@{commit[:12]}")


+ git -C /content/embedding-kd remote set-url origin https://github.com/Savoxism/embedding-kd.git
+ git -C /content/embedding-kd reset --hard
+ git -C /content/embedding-kd fetch --prune origin
+ git -C /content/embedding-kd checkout -B nqd_mass_geom_loss origin/nqd_mass_geom_loss
+ git -C /content/embedding-kd reset --hard origin/nqd_mass_geom_loss
+ git -C /content/embedding-kd pull --ff-only origin nqd_mass_geom_loss
Ready: nqd_mass_geom_loss@b1f683b299d6


In [31]:
# 3. Cài/đồng bộ dependencies theo code vừa pull
import os
import sys

os.chdir(REPO_DIR)
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-r", str(REPO_DIR / "requirements.txt")],
    check=True,
)
print("Dependencies are ready.")


Dependencies are ready.


In [32]:
# 4. Resolve model pair, GPU và nơi lưu artifacts
import torch

PAIR_CONFIGS = {
    "qwen3_0_6b_to_minilmv2_h384": {
        "teacher": "Qwen/Qwen3-Embedding-0.6B",
        "student": "nreimers/MiniLMv2-L6-H384-distilled-from-BERT-Base",
        "pooling": "last_token",
    },
    "bge_m3_to_minilmv2_h768": {
        "teacher": "BAAI/bge-m3",
        "student": "nreimers/MiniLMv2-L6-H768-distilled-from-BERT-Base",
        "pooling": "cls",
    },
    "qwen3_4b_to_bert_base": {
        "teacher": "Qwen/Qwen3-Embedding-4B",
        "student": "google-bert/bert-base-uncased",
        "pooling": "last_token",
    },
}
pair = PAIR_CONFIGS[PAIR_KEY]

if REQUIRE_GPU and not torch.cuda.is_available():
    raise RuntimeError("Không tìm thấy CUDA GPU. Trong Colab chọn Runtime → Change runtime type → GPU.")
if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    vram_gb = props.total_memory / 2**30
    print(f"GPU: {props.name} ({vram_gb:.1f} GiB)")
    if PAIR_KEY == "qwen3_4b_to_bert_base" and vram_gb < 35:
        print("WARNING: Qwen3-4B ở cấu hình hiện tại có thể OOM trên GPU dưới khoảng 35 GiB.")

if USE_GOOGLE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    artifact_root = Path(DRIVE_ROOT)
else:
    artifact_root = REPO_DIR

# cache_dir không mang RUN_TAG: teacher embeddings và HeatGeo graph không phụ
# thuộc row mode, nên ba variant dùng chung một cache và chỉ build graph một lần.
cache_dir = artifact_root / "cache" / "heatgeo" / PAIR_KEY
log_dir = artifact_root / "logs" / "heatgeo" / PAIR_KEY
save_dir = artifact_root / "models" / "heatgeo" / PAIR_KEY / RUN_TAG
weights_dir = artifact_root / "models" / "heatgeo_weights" / PAIR_KEY / RUN_TAG
for path in (cache_dir, log_dir, save_dir, weights_dir):
    path.mkdir(parents=True, exist_ok=True)

print(f"Teacher: {pair['teacher']}")
print(f"Student: {pair['student']}")
print(f"Pooling: {pair['pooling']}")
print(f"Objective: L_rel + {ROW_WEIGHT} * L_row")
print(f"Row starts at epoch: {ROW_START_EPOCH}")
print(f"Outputs: {save_dir}")


GPU: NVIDIA RTX PRO 6000 Blackwell Server Edition (95.0 GiB)
Teacher: Qwen/Qwen3-Embedding-0.6B
Student: nreimers/MiniLMv2-L6-H384-distilled-from-BERT-Base
Pooling: last_token
Objective: L_rel + 1.0 * L_row
Row starts at epoch: 1
Outputs: /content/embedding-kd/models/heatgeo/qwen3_0_6b_to_minilmv2_h384/row_loss_run_closure_ht_amb_w1_e1_seed42


In [33]:
# 5. Train HeatGeo
import shlex

command = [
    sys.executable, "main.py",
    "--method", "heatgeo",
    "--train_data", TRAIN_DATA,
    "--student_model", pair["student"],
    "--teacher_model", pair["teacher"],
    "--pooling_method", pair["pooling"],
    "--row_weight", str(ROW_WEIGHT),
    "--row_start_epoch", str(ROW_START_EPOCH),
    "--batch_size", str(BATCH_SIZE),
    "--epochs", str(EPOCHS),
    "--lr", str(LEARNING_RATE),
    "--max_length", str(MAX_LENGTH),
    "--seed", str(SEED),
    "--num_workers", str(NUM_WORKERS),
    "--cache_path", str(cache_dir / "teacher_train.pt"),
    "--heatgeo_cache_path", str(cache_dir / "graph.pt"),
    "--heatgeo_log_dir", str(log_dir),
    "--save_dir", str(save_dir),
    "--weights_dir", str(weights_dir),
    "--wandb_project", WANDB_PROJECT,
    "--wandb_run_name", f"{PAIR_KEY}_{RUN_TAG}",
    "--wandb_mode", WANDB_MODE,
]
if FINAL_WEIGHTS_ONLY:
    command.append("--final_weights_only")
if not USE_WANDB:
    command.append("--no_wandb")
if EXTRA_ARGS.strip():
    command.extend(shlex.split(EXTRA_ARGS))

env = os.environ.copy()
env["TOKENIZERS_PARALLELISM"] = "false"
env["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
print("+", shlex.join(command))
process = subprocess.Popen(
    command,
    cwd=REPO_DIR,
    env=env,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)
for line in process.stdout:
    print(line, end="")
return_code = process.wait()
if return_code != 0:
    raise RuntimeError(f"Training failed with exit code {return_code}")


+ /usr/bin/python3 main.py --method heatgeo --train_data data/train_set/merged_3_data_5k_each.csv --student_model nreimers/MiniLMv2-L6-H384-distilled-from-BERT-Base --teacher_model Qwen/Qwen3-Embedding-0.6B --pooling_method last_token --row_mode closure_ht --row_ambient 1 --row_weight 1.0 --row_start_epoch 1 --batch_size 64 --epochs 5 --lr 2e-05 --max_length 256 --seed 42 --num_workers 4 --cache_path /content/embedding-kd/cache/heatgeo/qwen3_0_6b_to_minilmv2_h384/teacher_train.pt --heatgeo_cache_path /content/embedding-kd/cache/heatgeo/qwen3_0_6b_to_minilmv2_h384/graph.pt --heatgeo_log_dir /content/embedding-kd/logs/heatgeo/qwen3_0_6b_to_minilmv2_h384 --save_dir /content/embedding-kd/models/heatgeo/qwen3_0_6b_to_minilmv2_h384/row_loss_run_closure_ht_amb_w1_e1_seed42 --weights_dir /content/embedding-kd/models/heatgeo_weights/qwen3_0_6b_to_minilmv2_h384/row_loss_run_closure_ht_amb_w1_e1_seed42 --wandb_project iclr-mdd-heatgeo --wandb_run_name qwen3_0_6b_to_minilmv2_h384_row_loss_run_clos

In [34]:
# 6. Xem artifacts và các metrics cuối
import json

print("Checkpoints:", save_dir)
print("Weights:", weights_dir)
metrics_path = save_dir / "metrics.jsonl"
if metrics_path.exists():
    records = [json.loads(line) for line in metrics_path.read_text().splitlines() if line.strip()]
    print(json.dumps(records[-1], indent=2, ensure_ascii=False) if records else "metrics.jsonl is empty")
else:
    print("Không tìm thấy metrics.jsonl")


Checkpoints: /content/embedding-kd/models/heatgeo/qwen3_0_6b_to_minilmv2_h384/row_loss_run_closure_ht_amb_w1_e1_seed42
Weights: /content/embedding-kd/models/heatgeo_weights/qwen3_0_6b_to_minilmv2_h384/row_loss_run_closure_ht_amb_w1_e1_seed42
{
  "method": "heatgeo",
  "seed": 42,
  "test": {
    "avg": 74.62,
    "avg_in": 67.93,
    "avg_out": 77.96,
    "classification": {
      "data/test_set/banking77_test.csv": {
        "accuracy": 0.8800390117035111,
        "f1": 0.8804239509400739
      },
      "data/test_set/emotion_test.csv": {
        "accuracy": 0.7139979859013091,
        "f1": 0.624819713581902
      },
      "data/test_set/tweet_test.csv": {
        "accuracy": 0.7042540792540792,
        "f1": 0.7084223479327911
      }
    },
    "pair": {
      "data/test_set/mrpc_test.csv": {
        "accuracy": 0.7043478260869566,
        "average_precision": 0.8219784488699756,
        "best_threshold": 0.9296482412060302,
        "f1": 0.6374473325025221,
        "precision": 0.